In [ ]:
# Para cargar todos los datos
import os
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

npz_folder = 'Insecta/NPZ/'

X_total = []
y_total = []
# Recorremos todos los archivos .npz en el directorio
for file in tqdm(os.listdir(npz_folder)):
    if file.endswith('.npz'):
        path = os.path.join(npz_folder, file)
        try:
            data = np.load(path)
            spec = data['spec']
            label = data['label'].item() 

            # Solo guardamos espectrogramas con forma exacta (1025, 313)
            if spec.shape == (1025, 313):
                X_total.append(spec) #Espectograma 
                y_total.append(label) #Etiqueta asociada a ese espectograma (Nombre de la especie) 
            else:
                print(f"Forma inválida en: {file} -> {spec.shape}")
        except Exception as e:
            print(f"Error leyendo {file}: {e}")

# Se cargaron
print(f"\nEspectrogramas válidos cargados: {len(X_total)}")

In [ ]:
# Para entrenamiento
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight

# Convertir a arrays
X = np.array(X_total)
y = np.array(y_total)

# Filtrar clases con menos de 2 muestras
label_counts = Counter(y)
clases_validas = [label for label, count in label_counts.items() if count >= 2]

X = np.array([x for x, label in zip(X, y) if label in clases_validas])
y = np.array([label for label in y if label in clases_validas])

print(f"Clases restantes tras filtrar: {set(y)}")
print("Distribución de clases:", label_counts)

# Codificar etiquetas
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_cat = to_categorical(y_encoded)

# MANEJO DE DESBALANCEO
# 1. Calcular pesos de clases
class_weights = compute_class_weight(
    'balanced', 
    classes=np.unique(y_encoded), 
    y=y_encoded
)
class_weights_dict = dict(enumerate(class_weights))
print("Pesos de clases:", class_weights_dict)

# 2. Aumento de datos (Data Augmentation)
from keras.preprocessing.image import ImageDataGenerator

# Crear generador de aumento de datos para espectrogramas
datagen = ImageDataGenerator(
    rotation_range=5,         # Rotación aleatoria (±5 grados)
    width_shift_range=0.05,   # Desplazamiento horizontal (5%)
    height_shift_range=0.05,  # Desplazamiento vertical (5%)
    zoom_range=0.1,           # Zoom aleatorio (±10%)
    fill_mode='constant'      # Rellenar con 0 (fondo)
)

# Normalizar y reshaping para CNN
X = X / np.max(X)  # normalización
X = X[..., np.newaxis]  # añadir canal

# Separar entrenamiento y validación
X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat, test_size=0.2, stratify=y_encoded, random_state=42
) 

In [ ]:
# MEJORAS EN LA ARQUITECTURA
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, BatchNormalization, GlobalAveragePooling2D, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(1025, 313, 1)),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.3),
    
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.4),
    
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.5),
    
    # Usar GlobalAveragePooling en lugar de Flatten
    GlobalAveragePooling2D(),
    
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(y_cat.shape[1], activation='softmax')
])

model.compile(
    optimizer='adam', 
    loss='categorical_crossentropy', 
    metrics=['accuracy', 
             'Precision', 
             'Recall',  # Importante para clases minoritarias
             'AUC']     # Útil para desbalanceo
)
model.summary()

In [ ]:
# ENTRENAMIENTO CON MANEJO DE DESBALANCEO
from keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint('best_model_insectos.h5', save_best_only=True)
]

# Entrenar con aumento de datos y pesos de clase
batch_size = 16
epochs = 50

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=batch_size),  # Aumento de datos
    steps_per_epoch=len(X_train) // batch_size,
    validation_data=(X_val, y_val),
    epochs=epochs,
    class_weight=class_weights_dict,  # Pesos de clases
    callbacks=callbacks
)


In [ ]:

# VISUALIZACIÓN Y EVALUACIÓN

# 1. Gráficas de entrenamiento
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Exactitud por Época')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida por Época')
plt.legend()
plt.savefig('training_history.png')
plt.show()

# 2. Reporte de clasificación
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred = model.predict(X_val)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_val, axis=1)

print("\nReporte de Clasificación:")
print(classification_report(y_true, y_pred_classes, target_names=le.classes_))

# 3. Matriz de confusión
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_true, y_pred_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicho')
plt.ylabel('Verdadero')
plt.title('Matriz de Confusión')
plt.savefig('confusion_matrix.png')
plt.show()